# Section 2: Tokenization — From Text to Numbers

Neural networks operate on numbers. Text is symbols. The bridge between them is **tokenization**: a procedure that splits a string into a list of *tokens* and assigns each token an integer ID.

We use a **character-level tokenizer** here — the simplest possible kind. Every unique character in the text gets its own integer ID. This keeps the vocabulary tiny and everything human-readable, so we can focus on the architecture rather than the tokenizer details.

## The `CharTokenizer` class

The tokenizer has three responsibilities:
1. **Build a vocabulary** — find every unique character in the corpus and sort them.
2. **`stoi` (string → int)** — a dict that maps each character to a unique integer ID.
3. **`itos` (int → string)** — the reverse dict, used to convert predictions back to text.

The two methods `encode` and `decode` use these dicts to convert between text and integers.

In [1]:
class CharTokenizer:
    def __init__(self, corpus: str):
        # Step 1: find every unique character that appears in the corpus.
        # `sorted` makes the mapping deterministic — without it, the IDs would
        # change every time we create a new tokenizer from the same text.
        unique_chars = sorted(set(corpus))

        # Step 2: build two lookup tables.
        # `stoi` (string-to-integer): given a character, return its ID.
        # `itos` (integer-to-string): given an ID, return its character.
        # These are just Python dicts — very fast O(1) lookups.
        self.stoi = {ch: i for i, ch in enumerate(unique_chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

        # Step 3: record the vocabulary size.
        # The embedding matrix we build later will need one row per token,
        # so it needs to know how many unique tokens exist.
        self.vocab_size = len(unique_chars)

    def encode(self, text: str) -> list[int]:
        # Convert each character to its integer ID using `stoi`.
        # The model will receive this list of integers — never the raw text.
        return [self.stoi[ch] for ch in text]

    def decode(self, ids: list[int]) -> str:
        # Reverse: convert each integer ID back to its character using `itos`,
        # then join the characters into a string.
        return "".join(self.itos[i] for i in ids)

## Step 1: Build the vocabulary

We instantiate the tokenizer on our sentence. Internally it finds all unique characters, sorts them, and assigns IDs 0, 1, 2, … in alphabetical order.

In [2]:
sentence = "the cat sat on the mat"
tok = CharTokenizer(sentence)

print(f"Input sentence : {sentence!r}")
print(f"Unique chars   : {sorted(tok.stoi.keys())}")
print(f"Vocabulary size: {tok.vocab_size}")

Input sentence : 'the cat sat on the mat'
Unique chars   : [' ', 'a', 'c', 'e', 'h', 'm', 'n', 'o', 's', 't']
Vocabulary size: 10


## Step 2: `stoi` — character → integer ID

Each unique character gets a unique integer label, assigned in alphabetical order. Space (`' '`) sorts first, so it gets ID 0.

In [3]:
print("stoi (character -> integer ID):")
for ch, idx in tok.stoi.items():
    print(f"  {ch!r:4s} -> {idx}")

stoi (character -> integer ID):
  ' '  -> 0
  'a'  -> 1
  'c'  -> 2
  'e'  -> 3
  'h'  -> 4
  'm'  -> 5
  'n'  -> 6
  'o'  -> 7
  's'  -> 8
  't'  -> 9


## Step 3: `itos` — integer ID → character

The exact reverse of `stoi`. This is what the model uses after training to convert its predicted integer IDs back into readable text.

In [4]:
print("itos (integer ID -> character):")
for idx, ch in tok.itos.items():
    print(f"  {idx} -> {ch!r}")

itos (integer ID -> character):
  0 -> ' '
  1 -> 'a'
  2 -> 'c'
  3 -> 'e'
  4 -> 'h'
  5 -> 'm'
  6 -> 'n'
  7 -> 'o'
  8 -> 's'
  9 -> 't'


## Step 4: `encode` — text → list of integers

The model will only ever see these integers, never the raw characters. Notice that every `'t'` in the sentence maps to the same ID (9), and every `'a'` maps to 1.

In [5]:
ids = tok.encode(sentence)

print(f"encode({sentence!r})")
print(f"  -> {ids}")
print()
# Show the character-by-character mapping side by side
print("Character-by-character breakdown:")
for ch, idx in zip(sentence, ids):
    print(f"  {ch!r:4s} -> {idx}")

encode('the cat sat on the mat')
  -> [9, 4, 3, 0, 2, 1, 9, 0, 8, 1, 9, 0, 7, 6, 0, 9, 4, 3, 0, 5, 1, 9]

Character-by-character breakdown:
  't'  -> 9
  'h'  -> 4
  'e'  -> 3
  ' '  -> 0
  'c'  -> 2
  'a'  -> 1
  't'  -> 9
  ' '  -> 0
  's'  -> 8
  'a'  -> 1
  't'  -> 9
  ' '  -> 0
  'o'  -> 7
  'n'  -> 6
  ' '  -> 0
  't'  -> 9
  'h'  -> 4
  'e'  -> 3
  ' '  -> 0
  'm'  -> 5
  'a'  -> 1
  't'  -> 9


## Step 5: `decode` — integers → text (round-trip check)

`decode` is the inverse of `encode`. If we encode a string and then decode the result, we should get the original string back exactly. This **round-trip invariant** is the basic correctness check for any tokenizer.

In [6]:
recovered = tok.decode(ids)

print(f"decode({ids})")
print(f"  -> {recovered!r}")
print()
print(f"Round-trip check — decode(encode(sentence)) == sentence: {recovered == sentence}")

decode([9, 4, 3, 0, 2, 1, 9, 0, 8, 1, 9, 0, 7, 6, 0, 9, 4, 3, 0, 5, 1, 9])
  -> 'the cat sat on the mat'

Round-trip check — decode(encode(sentence)) == sentence: True
